# 1일차 8차시 · 음성 회의 Agent 통합 소프트웨어 실습

이 노트북은 홈페이지를 구경하는 실습이 아닙니다. 실제 합성 WAV를 넣고 다음 프로그램을 끝까지 실행합니다.

`WAV → faster-whisper → STT 품질검사 → LangChain/Ollama → LangGraph 사람 승인 → local trace → READY/HOLD`

실제 회사 회의 음성 대신 제공된 합성 데이터만 사용합니다. 자동 메일 발송이나 외부 게시 기능은 실행하지 않습니다.

## 0. 저장소와 Python Kernel 확인

In [1]:
from pathlib import Path
import json, os, platform, sys

START = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (START, *START.parents) if (candidate / 'src').exists()),
    None,
)
assert ROOT is not None, '저장소 최상위 폴더를 찾지 못했습니다.'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({'python': platform.python_version(), 'root': str(ROOT)})
if sys.version_info < (3, 12):
    raise RuntimeError('Python 3.12 Kernel을 선택하세요.')

{'python': '3.12.12', 'root': '/Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation'}


## 1. 필요한 library 설치

이미 설치된 library는 다시 설치하지 않습니다. STT나 Ollama 설치가 실패해도 텍스트 fixture와 fixture LLM으로 후속 흐름을 계속할 수 있습니다.

In [2]:
from importlib.util import find_spec
import subprocess

def ensure_requirements(requirements_file, modules, *, required):
    missing = [name for name in modules if find_spec(name) is None]
    if not missing:
        print({'requirements': requirements_file, 'status': 'ALREADY_READY'})
        return True
    command = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / requirements_file)]
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    status = 'INSTALLED' if completed.returncode == 0 else 'INSTALL_FAILED'
    print({'requirements': requirements_file, 'missing': missing, 'status': status})
    if completed.returncode != 0 and required:
        raise RuntimeError(completed.stderr.strip().splitlines()[-1])
    return completed.returncode == 0

ensure_requirements('requirements-day1.txt', ['langchain_core', 'langgraph', 'pydantic'], required=True)
STT_READY = ensure_requirements('requirements-stt-optional.txt', ['faster_whisper'], required=False)
OLLAMA_LIBRARY_READY = ensure_requirements(
    'requirements-local-llm-optional.txt', ['langchain_ollama'], required=False
)

{'requirements': 'requirements-day1.txt', 'status': 'ALREADY_READY'}
{'requirements': 'requirements-stt-optional.txt', 'status': 'ALREADY_READY'}
{'requirements': 'requirements-local-llm-optional.txt', 'status': 'ALREADY_READY'}


## 2. 실행 전 점검

- `demo_meeting.wav`: 약 66초 합성 한국어 회의 음성
- `small`: 품질 실습용 모델
- `local_files_only=True`: 수업 중 모델을 새로 내려받지 않음
- `qwen3:4b`: 설치돼 있으면 실제 Ollama, 아니면 fixture로 복구

In [3]:
from src.ollama_tool_agent import probe_ollama

audio_path = ROOT / 'data/demo_meeting.wav'
fixture_path = ROOT / 'data/demo_meeting_transcript.txt'
ollama_status = probe_ollama()
provider = 'ollama' if OLLAMA_LIBRARY_READY and ollama_status['model_ready'] else 'fixture'
print(json.dumps({
    'audio_exists': audio_path.exists(),
    'audio_size_mb': round(audio_path.stat().st_size / 1024 / 1024, 1),
    'stt_library_ready': STT_READY,
    'ollama': ollama_status,
    'selected_provider': provider,
}, ensure_ascii=False, indent=2))

{
  "audio_exists": true,
  "audio_size_mb": 2.8,
  "stt_library_ready": true,
  "ollama": {
    "cli_installed": true,
    "cli_path": "/usr/local/bin/ollama",
    "server_ready": true,
    "version": "0.32.15",
    "models": [
      "qwen3:4b"
    ],
    "model_ready": true,
    "error_code": null,
    "detail": null,
    "recommended_lane": "ollama"
  },
  "selected_provider": "ollama"
}


## 3. 음성부터 LangGraph 승인까지 한 번에 실행

STT가 `READY`면 요약 검토에서 한 번 멈춥니다. STT가 `HOLD`면 전사문 검토와 요약 검토에서 두 번 멈춥니다. 여기서는 코드를 따라가기 위해 사람의 선택을 함수 인자로 넣지만, 내부에서는 실제 `interrupt()`와 `Command(resume=...)`가 실행됩니다.

In [4]:
from src.meeting_agent_workflow import run_meeting_agent_workflow

result = run_meeting_agent_workflow(
    audio_path=audio_path,
    transcript_fixture_path=fixture_path,
    output_dir=ROOT / 'output/notebook-meeting-agent',
    model_size='small',
    device='cpu',
    compute_type='int8',
    language='ko',
    beam_size=5,
    local_files_only=True,
    provider=provider,
    llm_model='qwen3:4b' if provider == 'ollama' else None,
    transcript_decision='accept',
    summary_decision='approve',
)
print(json.dumps({
    'stt_mode': result['stt']['mode'],
    'stt_quality_gate': result['stt']['quality_gate']['decision'],
    'stt_reasons': result['stt']['quality_gate']['reasons'],
    'provider_used': result['langgraph']['final_state'].get('provider_used'),
    'provider_fallback_reason': result['langgraph']['final_state'].get('provider_fallback_reason'),
    'interrupt_stages': [item['stage'] for item in result['langgraph']['interruptions']],
    'graph_status': result['langgraph']['final_state']['status'],
    'release_gate': result['evaluation']['decision'],
}, ensure_ascii=False, indent=2))

{
  "stt_mode": "local_stt",
  "stt_quality_gate": "READY",
  "stt_reasons": [],
  "provider_used": "ollama",
  "provider_fallback_reason": null,
  "interrupt_stages": [
    "summary_review"
  ],
  "graph_status": "READY_FOR_EXPORT",
  "release_gate": "READY"
}


## 4. STT 품질 증거 확인

문장만 읽지 말고 언어 확률, VAD 적용 전후 길이, segment 수, 품질 flag를 확인합니다. `HIGH_NO_SPEECH_PROBABILITY`는 no-speech와 낮은 log probability가 함께 나타날 때만 기록합니다.

In [5]:
quality = result['stt']['quality_gate']
metadata = result['stt']['transcription_metadata']
flagged = [
    {
        'id': segment['id'],
        'start': segment.get('start'),
        'end': segment.get('end'),
        'flags': segment.get('quality_flags', []),
        'word_probability': segment.get('average_word_probability'),
    }
    for segment in result['stt']['segments']
    if segment.get('quality_flags')
]
print(json.dumps({'metadata': metadata, 'quality_gate': quality, 'flagged': flagged[:5]}, ensure_ascii=False, indent=2))

{
  "metadata": {
    "language": "ko",
    "language_probability": 1.0,
    "duration_seconds": 66.45,
    "duration_after_vad_seconds": 66.25,
    "model": "small",
    "device": "cpu",
    "compute_type": "int8",
    "beam_size": 5,
    "vad_filter": true,
    "word_timestamps": true,
    "condition_on_previous_text": false
  },
  "quality_gate": {
    "decision": "READY",
    "flagged_segment_count": 0,
    "reasons": [],
    "human_decision_required": true,
    "detected_language": "ko",
    "language_probability": 1.0,
    "segment_count": 14,
    "text_length": 415
  },
  "flagged": []
}


## 5. LangGraph State와 사람 승인 이력 확인

In [6]:
final_state = result['langgraph']['final_state']
print(json.dumps({
    'interruptions': result['langgraph']['interruptions'],
    'transcript_review': final_state.get('transcript_review'),
    'summary_review': final_state.get('summary_review'),
    'audit_events': final_state.get('audit_events'),
    'status': final_state.get('status'),
    'automatic_email': final_state.get('automatic_email'),
}, ensure_ascii=False, indent=2))

{
  "interruptions": [
    {
      "stage": "summary_review",
      "question": "회의 요약과 할 일을 승인·수정·거절하시겠습니까?",
      "summary": "고객문의 자동화 폭의 첫 범위를 정하는 회의에서, 배송지연 문의와 만품 절차 안을 1차 범위로 제안하며, 외부 발행 전에는 반드시 사람이 승인해야 한다는 결정사항을 도출했습니다.",
      "action_items": [
        {
          "task": "1차 범위 정의 및 공개",
          "owner": "화자미상",
          "due_date": "2026-08-27",
          "evidence_ids": [
            "s01",
            "s03",
            "s04",
            "s05"
          ]
        },
        {
          "task": "2026년 샘플 정리",
          "owner": "화자미상",
          "due_date": "2026-08-28",
          "evidence_ids": [
            "s06",
            "s07",
            "s08"
          ]
        },
        {
          "task": "근거 문장 검토 및 정책 모호성 확인",
          "owner": "화자미상",
          "due_date": "2026-08-28",
          "evidence_ids": [
            "s09"
          ]
        },
        {
          "task": "자동 매일 발송 대신 초안 프리뷰, 귀승인, 수정 거절 중 하나 선택",
          "owner": "화자미상",
          "due_da

## 6. 실패 경로 실행 · 음성이 없으면 HOLD 후 사람 확인

없는 음성 경로를 주면 transcript fixture로 복구합니다. 기술 실행은 계속되지만 `STT_FALLBACK_USED` 때문에 반드시 사람 검토를 거칩니다.

In [7]:
fallback_result = run_meeting_agent_workflow(
    audio_path=ROOT / 'data/missing_audio.wav',
    transcript_fixture_path=fixture_path,
    output_dir=ROOT / 'output/notebook-meeting-agent-fallback',
    provider='fixture',
    transcript_decision='accept',
    summary_decision='approve',
)
print(json.dumps({
    'stt_mode': fallback_result['stt']['mode'],
    'quality_gate': fallback_result['stt']['quality_gate'],
    'interrupt_stages': [item['stage'] for item in fallback_result['langgraph']['interruptions']],
    'release_gate': fallback_result['evaluation']['decision'],
}, ensure_ascii=False, indent=2))

{
  "stt_mode": "fixture",
  "quality_gate": {
    "decision": "HOLD",
    "flagged_segment_count": 0,
    "reasons": [
      "STT_FALLBACK_USED"
    ],
    "human_decision_required": true,
    "detected_language": "ko",
    "language_probability": null,
    "segment_count": 6,
    "text_length": 448
  },
  "interrupt_stages": [
    "transcript_review",
    "summary_review"
  ],
  "release_gate": "READY"
}


## 7. 생성 파일과 좁은 테스트 확인

In [8]:
for label, path in result['outputs'].items():
    print(label, Path(path).exists(), path)

completed = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_meeting_agent_workflow.py'],
    capture_output=True, text=True, check=False,
)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr)

transcript_text True /Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation/output/notebook-meeting-agent/stt/transcript.txt
transcript_json True /Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation/output/notebook-meeting-agent/stt/transcript.json
meeting_result True /Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation/output/notebook-meeting-agent/stt/meeting_result.json
trace True /Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation/output/notebook-meeting-agent/trace.json
workflow_result True /Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation/output/notebook-meeting-agent/workflow_result.json


......                                                                   [100%]
6 passed in 0.32s



## 완료 체크

- [ ] `stt_mode=local_stt` 또는 설계된 `fixture` 복구를 확인했다.
- [ ] STT `READY/HOLD` 이유를 읽었다.
- [ ] `provider_used`와 `provider_fallback_reason`을 확인했다.
- [ ] LangGraph interrupt 단계와 audit event를 확인했다.
- [ ] `automatic_email=false`와 최종 `READY/HOLD`를 확인했다.
- [ ] transcript·trace·workflow JSON 파일을 열어봤다.